In [1]:
"""
02_verificacion_airbus.py

Este script realiza la verificación oficial de los cálculos matemáticos descritos
en la sección "CHECK" del "AIRBUS Quantum Computing Challenge".

Su objetivo es comprobar que las ecuaciones de la dinámica de vuelo, el cálculo
atmosférico y la función de costo (Phi) están correctamente implementadas,
comparando los resultados computados en Python puro (sin Numba) con los
valores esperados provistos por Airbus.

Ideal para validación de la lógica matemática previa a la optimización cuántica.

Autor: [Marlio Erazo]
"""

import math

# =============================================================================
# CONSTANTES FÍSICAS Y AERODINÁMICAS (Airbus A320)
# =============================================================================
Cx_0    = 0.014
k       = 0.09
S_REF   = 120.0                     # m^2
η       = 0.06 / 3600.0             # kg/(N.s) - Conversión de kg/(N.h)
g_0     = 9.80665                   # m/s^2
CI      = 30.0 / 60.0               # kg/s - Conversión de kg/min (Cost Index)

Ts_0    = 288.15                    # K
ρ_0     = 1.225                     # kg/m^3
L_z     = -0.0065                   # K/m
R       = 287.05287                 # N.m/(kg.K)
α_0     = -g_0 / (R * L_z)

# Fase de crucero final
s_F     = 400000.0                  # m (L = 400 km)
Zp_F_ft = 36000.0                   # ft
Zp_F_m  = Zp_F_ft * 0.3048          # m
M_CRZ   = 0.80                      # Mach de crucero


# =============================================================================
# FUNCIÓN DE VERIFICACIÓN (Sección "CHECK" de Airbus)
# =============================================================================

def verificar_estado_final():
    # -------------------------------------------------------------------------
    # 1. INPUTS PROVISTOS POR EL DOCUMENTO (Variables en el estado N-1)
    # -------------------------------------------------------------------------
    Zp_N1_ft = 36000.0              # ft
    v_N1     = 223.61               # m/s
    v_F      = 236.15               # m/s (Mach = 0.8)
    m_N1     = 59042.0              # kg
    t_N1     = 880.8                # s
    s_N1     = 168717.2             # m
    lam_N1   = 1.0                  # λ (Thrust ratio)

    # -------------------------------------------------------------------------
    # 2. CÁLCULOS MATEMÁTICOS (Paso a paso según el paper)
    # -------------------------------------------------------------------------

    # Thrust Máximo en Ascenso (Nota: la ecuación toma Zp en ft)
    F_N_MCL_N1 = 140000.0 - 2.53 * Zp_N1_ft

    # Densidad final (rho_F)
    rho_F = ρ_0 * ((Ts_0 + L_z * Zp_F_m) / Ts_0)**(α_0 - 1.0)

    # Constantes A, B, C de la ecuación cuadrática
    term_A1 = -(rho_F * S_REF * Cx_0) / (2.0 * m_N1)
    term_A2 = (6.0 * k * m_N1 * g_0**2) / (rho_F * S_REF * v_N1**4)
    A = term_A1 - term_A2

    B = (16.0 * k * m_N1 * g_0**2) / (rho_F * S_REF * v_N1**3)

    term_C1 = F_N_MCL_N1 / m_N1
    term_C2 = (12.0 * k * m_N1 * g_0**2) / (rho_F * S_REF * v_N1**2)
    C = term_C1 - term_C2

    # Discriminante D
    D = math.sqrt(B**2 - 4.0 * A * C)

    # Cálculos de tiempo, masa y distancia en el punto B
    arg_tanh1 = (2.0 * A * v_N1 + B) / D
    arg_tanh2 = (2.0 * A * v_F + B) / D

    t_B = t_N1 + (2.0 / D) * (math.atanh(arg_tanh1) - math.atanh(arg_tanh2))
    m_B = m_N1 - η * lam_N1 * F_N_MCL_N1 * (t_B - t_N1)

    arg_log = (D - 2.0 * A * v_F - B) / (D - 2.0 * A * v_N1 - B)
    s_B = s_N1 + (1.0 / A) * math.log(arg_log) - ((B + D) / (2.0 * A)) * (t_B - t_N1)

    # Cálculos en el punto F (Final)
    exponente_mF = (-2.0 * η * g_0 * math.sqrt(k * Cx_0) / v_F) * (s_F - s_B)
    m_F = m_B * math.exp(exponente_mF)

    t_F = t_B + (s_F - s_B) / v_F

    # Función de costo final (Phi)
    phi = -m_F + CI * (t_B - s_B / v_F)

    # -------------------------------------------------------------------------
    # 3. VALORES ESPERADOS (Tomados textualmente del paper de Airbus)
    # -------------------------------------------------------------------------
    esperado = {
        "F_N_MCL_N1": 48920.0,
        "rho_F"     : 0.3652,
        "A"         : -3.31814e-05,
        "B"         : 0.0166878,
        "C"         : -1.970107,
        "t_B"       : 992.839,
        "m_B"       : 58950.65,
        "s_B"       : 194453.96,
        "m_F"       : 58358.27,
        "t_F"       : 1863.24,
        "Phi"       : -58273.65
    }

    calculado = {
        "F_N_MCL_N1": F_N_MCL_N1,
        "rho_F"     : rho_F,
        "A"         : A,
        "B"         : B,
        "C"         : C,
        "t_B"       : t_B,
        "m_B"       : m_B,
        "s_B"       : s_B,
        "m_F"       : m_F,
        "t_F"       : t_F,
        "Phi"       : phi
    }

    # -------------------------------------------------------------------------
    # 4. PRESENTACIÓN DE RESULTADOS
    # -------------------------------------------------------------------------
    print("\n" + "="*75)
    print("  VERIFICACIÓN DE MODELO FÍSICO A320 - SECCIÓN 'CHECK' DE AIRBUS")
    print("="*75)
    print(f"{'Variable':<15} | {'Valor Calculado (Python)':<25} | {'Valor Esperado (Doc)':<20}")
    print("-" * 75)

    for var in esperado.keys():
        calc_val = calculado[var]
        esp_val  = esperado[var]

        # Formateo dinámico para notación científica en A, o decimales fijos en los demás
        if var == "A":
            str_calc = f"{calc_val:.6e}"
            str_esp  = f"{esp_val:.6e}"
        else:
            str_calc = f"{calc_val:.6f}"
            str_esp  = f"{esp_val:.6f}"

        print(f"{var:<15} | {str_calc:<25} | {str_esp:<20}")

    print("="*75)

    # Comprobación de éxito general (Tolerancia muy pequeña por diferencias de redondeo del PDF)
    error_phi = abs((calculado["Phi"] - esperado["Phi"]) / esperado["Phi"]) * 100
    print(f"\n>> Diferencia porcentual en la Función de Costo (Phi): {error_phi:.6f} %")
    if error_phi < 0.01:
        print(">> ESTADO: ÉXITO [El modelo matemático coincide con Airbus]\n")
    else:
        print(">> ESTADO: ALERTA [Revisar discrepancias matemáticas]\n")


if __name__ == "__main__":
    verificar_estado_final()


  VERIFICACIÓN DE MODELO FÍSICO A320 - SECCIÓN 'CHECK' DE AIRBUS
Variable        | Valor Calculado (Python)  | Valor Esperado (Doc)
---------------------------------------------------------------------------
F_N_MCL_N1      | 48920.000000              | 48920.000000        
rho_F           | 0.365183                  | 0.365200            
A               | -3.318142e-05             | -3.318140e-05       
B               | 0.016688                  | 0.016688            
C               | -1.970107                 | -1.970107           
t_B             | 992.823403                | 992.839000          
m_B             | 58950.663586              | 58950.650000        
s_B             | 194450.247419             | 194453.960000       
m_F             | 58358.268977              | 58358.270000        
t_F             | 1863.243697               | 1863.240000         
Phi             | -58273.566459             | -58273.650000       

>> Diferencia porcentual en la Función de Costo (Phi)